<span style="font-size:24pt; color:blue; font-family: 'Times New Roman'">Notebook:: 
    <span style="color:green;"> To Preprocess the FACTS-FAIR input to include SRCities scenarios. </span>
</span>
<br>
<span style="font-size:12pt; color:black; font-family:Georgia, serif;font-style:italic">by Praveen Kumar & Kelly McCusker </span>
<br>

In [1]:
import numpy as np
import h5py
import xarray as xr
import os

Before proceeding, please request `fair_scenarios.h5` data from Chris Smith(smithc@iiasa.ac.at)

In [ ]:
data_path="_data/fair_scenarios.h5" 

with h5py.File(data_path, "r") as f:
    print("Top-level keys:", list(f.keys()))

Top-level keys: ['H', 'HL', 'L', 'LN', 'M', 'ML', 'VL']


In [3]:
groups = ["H", "HL", "L", "LN", "M", "ML", "VL"]

opened = {}
for g in groups:
    try:
        ds = xr.open_dataset(
            data_path,
            engine="h5netcdf",
            group=f"/{g}",
            phony_dims="access",   
        )
        opened[g] = ds
        print(f"\n--- {g} ---")
        print(ds)
    except Exception as e:
        print(f"\n--- {g} FAILED ---")
        print(e)


--- H ---
<xarray.Dataset> Size: 54MB
Dimensions:                              (phony_dim_0: 751, phony_dim_1: 2237)
Dimensions without coordinates: phony_dim_0, phony_dim_1
Data variables:
    aerosol_effective_radiative_forcing  (phony_dim_0, phony_dim_1) float64 13MB ...
    effective_radiative_forcing          (phony_dim_0, phony_dim_1) float64 13MB ...
    ocean_heat_content                   (phony_dim_0, phony_dim_1) float64 13MB ...
    temperature                          (phony_dim_0, phony_dim_1) float64 13MB ...

--- HL ---
<xarray.Dataset> Size: 54MB
Dimensions:                              (phony_dim_0: 751, phony_dim_1: 2237)
Dimensions without coordinates: phony_dim_0, phony_dim_1
Data variables:
    aerosol_effective_radiative_forcing  (phony_dim_0, phony_dim_1) float64 13MB ...
    effective_radiative_forcing          (phony_dim_0, phony_dim_1) float64 13MB ...
    ocean_heat_content                   (phony_dim_0, phony_dim_1) float64 13MB ...
    temperature       

### Extract to .nc

In [4]:
# # group -> scenario mapping
# group_scenarios = {
#     "H" : "ssp370",
#     "HL": "ssp585",
#     "M" : "ssp245",
#     "ML": "ssp245",
#     "L" :  "ssp245",
#     "VL": "ssp126",
#     "LN": "ssp245",
# }

# output_dir = "_data/cr8ed_fair_ip/"
# os.makedirs(output_dir, exist_ok=True)

# for group, scenario in group_scenarios.items():
#     # open one HDF5 group
#     ds = xr.open_dataset(
#         data_path,
#         engine="h5netcdf",
#         group=f"/{group}",
#         phony_dims="access",
#     )

#     print(f"\nProcessing SRC: {group} | scenario: {scenario}")
#     print(ds)

#     # infer dimensions for this group
#     dim0, dim1 = list(ds.dims)
#     n_years = ds.sizes[dim0]
#     n_samples = ds.sizes[dim1]

#     years = np.arange(1750, 1750 + n_years)
#     samples = np.arange(n_samples)

#     # ---------------------------
#     # 1) ocean_heat_content
#     # ---------------------------
#     ohc_ds = (
#         ds[["ocean_heat_content"]]
#         .rename_dims({dim0: "years", dim1: "samples"})
#         .assign_coords(years=years, samples=samples)
#         .expand_dims("locations")
#         .assign_coords(locations=[-1])
#         .transpose("samples", "years", "locations")
#         .astype("float64")
#         .assign_attrs(Scenario=scenario, SRC=group)
#     )

#     ohc_file = os.path.join(output_dir, f"src.{group}_{scenario}_ohc.nc")
#     ohc_ds.to_netcdf(ohc_file, engine="h5netcdf")
#     print(f"Saved {ohc_file}")

#     # ---------------------------
#     # 2) surface_temperature
#     # ---------------------------
#     temp_ds = (
#         ds[["temperature"]]
#         .rename_dims({dim0: "years", dim1: "samples"})
#         .rename_vars({"temperature": "surface_temperature"})
#         .assign_coords(years=years, samples=samples)
#         .expand_dims("locations")
#         .assign_coords(locations=[-1])
#         .transpose("samples", "years", "locations")
#         .astype("float64")
#         .assign_attrs(Scenario=scenario, SRC=group)
#     )

#     temp_file = os.path.join(output_dir, f"src.{group}_{scenario}_gsat.nc")
#     temp_ds.to_netcdf(temp_file, engine="h5netcdf")
#     print(f"Saved {temp_file}")

#     ds.close()

---

In [5]:
# years = temp_ds.years.values.flatten()
# climate_ds = xr.merge([temp_ds, ohc_ds]).squeeze("locations",drop=True).transpose("years", "samples")
# climate_file = os.path.join(output_dir, f"src.{group}_{scenario}_climate.nc")
# climate_ds.to_netcdf(climate_file, engine="h5netcdf", group=scenario, mode="w")
# yearsds = xr.Dataset({"year": years})
# yearsds.to_netcdf(climate_file, engine = 'h5netcdf', mode='a')

# print(climate_ds)
# print(f"Saved {climate_file}")

In [6]:
# group -> scenario mapping
group_scenarios = {
    "H" : "ssp370",
    "HL": "ssp585",
    "M" : "ssp245",
    "ML": "ssp245",
    "L" : "ssp245",
    "VL": "ssp126",
    "LN": "ssp245",
}

output_dir = "_data/cr8ed_fair_ip/"
os.makedirs(output_dir, exist_ok=True)

for group, scenario in group_scenarios.items():
    # open one HDF5 group
    ds = xr.open_dataset(
        data_path,
        engine="h5netcdf",
        group=f"/{group}",
        phony_dims="access",
    )

    print(f"\nProcessing SRC: {group} | scenario: {scenario}")
    print(ds)

    # infer dimensions for this group
    dim0, dim1 = list(ds.dims)
    n_years = ds.sizes[dim0]
    n_samples = ds.sizes[dim1]

    years = np.arange(1750, 1750 + n_years)
    samples = np.arange(n_samples)

    # ---------------------------
    # 1) ocean_heat_content
    # ---------------------------
    ohc_ds = (
        ds[["ocean_heat_content"]]
        .rename_dims({dim0: "years", dim1: "samples"})
        .assign_coords(years=years, samples=samples)
        .expand_dims("locations")
        .assign_coords(locations=[-1])
        .transpose("samples", "years", "locations")
        .astype("float64")
        .assign_attrs(Scenario=scenario, SRC=group)
    )

    ohc_file = os.path.join(output_dir, f"src.{group}_{scenario}_ohc.nc")
    ohc_ds.to_netcdf(ohc_file, engine="h5netcdf")
    print(f"Saved {ohc_file}")

    # ---------------------------
    # 2) surface_temperature
    # ---------------------------
    temp_ds = (
        ds[["temperature"]]
        .rename_dims({dim0: "years", dim1: "samples"})
        .rename_vars({"temperature": "surface_temperature"})
        .assign_coords(years=years, samples=samples)
        .expand_dims("locations")
        .assign_coords(locations=[-1])
        .transpose("samples", "years", "locations")
        .astype("float64")
        .assign_attrs(Scenario=scenario, SRC=group)
    )

    temp_file = os.path.join(output_dir, f"src.{group}_{scenario}_gsat.nc")
    temp_ds.to_netcdf(temp_file, engine="h5netcdf")
    print(f"Saved {temp_file}")

    # ---------------------------
    # 3) combined climate file
    # ---------------------------
    climate_ds = (
        xr.merge([temp_ds, ohc_ds])
        .squeeze("locations", drop=True)
        .transpose("years", "samples")
        .assign_attrs(Scenario=scenario, SRC=group)
    )

    climate_file = os.path.join(output_dir, f"src.{group}_{scenario}_climate.nc")
    climate_ds.to_netcdf(climate_file, engine="h5netcdf", group=scenario, mode="w")

    years_ds = xr.Dataset({"year": ("years", years)})
    years_ds.to_netcdf(climate_file, engine="h5netcdf", mode="a")

    print(climate_ds)
    print(f"Saved {climate_file}")

    ds.close()


Processing SRC: H | scenario: ssp370
<xarray.Dataset> Size: 54MB
Dimensions:                              (phony_dim_0: 751, phony_dim_1: 2237)
Dimensions without coordinates: phony_dim_0, phony_dim_1
Data variables:
    aerosol_effective_radiative_forcing  (phony_dim_0, phony_dim_1) float64 13MB ...
    effective_radiative_forcing          (phony_dim_0, phony_dim_1) float64 13MB ...
    ocean_heat_content                   (phony_dim_0, phony_dim_1) float64 13MB ...
    temperature                          (phony_dim_0, phony_dim_1) float64 13MB ...
Saved _data/cr8ed_fair_ip/src.H_ssp370_ohc.nc
Saved _data/cr8ed_fair_ip/src.H_ssp370_gsat.nc
<xarray.Dataset> Size: 27MB
Dimensions:              (samples: 2237, years: 751)
Coordinates:
  * samples              (samples) int64 18kB 0 1 2 3 4 ... 2233 2234 2235 2236
  * years                (years) int64 6kB 1750 1751 1752 ... 2498 2499 2500
Data variables:
    surface_temperature  (years, samples) float64 13MB 0.02783 0.02749 ... 5.477
 